# 3.5 Email Agent — proyecto del módulo 3 (adaptado a lab local)

**Qué se ve aquí:** combina todo el módulo en un agente:
- **State** `authenticated` + **context** con email y contraseña.
- **Dynamic tools** (`@wrap_model_call`): sin autenticar solo ve `authenticate`; autenticado ve `check_inbox` y `send_email`.
- **Dynamic prompt** según el estado de autenticación.
- **HITL** antes de `send_email`.

También se sirve con `langgraph dev` (`3.5_email_agent.py` + `langgraph.json`) y se prueba en **Agent Chat UI** (`agent-chat-ui/`).

**Convención de este fork:** 🔸 ORIGINAL DEL CURSO (comentada, con el motivo) → 🟢 ADAPTADO LOCAL (la que corre con Ollama). Celdas sin marca = iguales al curso.

### 📚 Teoría base del módulo 3: el ciclo del agente y dónde se engancha el middleware
`create_agent` es un **bucle**: el modelo piensa → (si pide tools) se ejecutan → el modelo vuelve a pensar → … hasta que responde sin pedir tools.

```
invoke ──► [before_agent] ──► ┌─► [before_model] ─► wrap_model_call( MODELO ) ─► [after_model] ─┐
                              │                                                                  │
                              │   ¿pidió tools? ── no ──► respuesta final ──► [after_agent] ──► fin
                              │        │ sí
                              └── wrap_tool_call( TOOL ) ◄┘
```
| Hook | Corre… | Ejemplo en este módulo |
|---|---|---|
| `before_agent` | 1 vez al empezar el `invoke` | borrar `ToolMessage`s (3.2) |
| `before_model` | antes de **cada** llamada al modelo | `SummarizationMiddleware` (3.2) |
| `wrap_model_call` | **envolviendo** cada llamada al modelo | cambiar modelo (3.4), tools (3.4, 3.5), prompt (`@dynamic_prompt` está hecho sobre esto) |
| `after_model` | después de cada respuesta del modelo | `HumanInTheLoopMiddleware` revisa los `tool_calls` y pausa (3.3) |
| `wrap_tool_call` | envolviendo cada ejecución de tool | reintentos o errores (como el interceptor MCP de M2.4) |

**Analogía:** el agente es un empleado; el middleware son los **supervisores** que revisan su trabajo en momentos fijos (al entrar, antes de hablar con el cliente, antes de enviar un correo…). El empleado no cambia; cambian las reglas que lo rodean.

### 📚 Teoría: el proyecto junta todo el módulo
| Pieza | Qué hace aquí | Lección |
|---|---|---|
| **Runtime context** `EmailContext` | Guarda el email y la contraseña "correctos" (en la vida real vendrían de un sistema seguro, nunca del prompt) | M2.2a |
| **State** `AuthenticatedState` | Campo `authenticated` que **cambia** cuando la tool `authenticate` acierta | M2.2b |
| **Tool que escribe el state** | `authenticate` devuelve `Command(update={"authenticated": True, "messages": [ToolMessage(...)]})` | M2.2b |
| **Dynamic tools** | Sin autenticar solo ve `authenticate`; autenticado ve `check_inbox` y `send_email` | 3.4 |
| **Dynamic prompt** | Cambia el system prompt según `authenticated` | 3.4 |
| **HITL** | Pausa antes de `send_email` | 3.3 |
| **Checkpointer** | Mantiene el state entre turnos y permite pausar y reanudar | 3.2 / 3.3 |

**El flujo:**
1. Usuario manda credenciales → el modelo solo ve `authenticate` → la llama → `authenticated=True`.
2. Siguiente llamada al modelo: el middleware ve `authenticated` → muestra `check_inbox` y `send_email` y cambia el prompt.
3. El modelo lee la bandeja y redacta una respuesta → pide `send_email` → **pausa (HITL)**.
4. El humano aprueba → se envía.

> **Seguridad:** la contraseña **nunca** la valida el modelo; la compara el **código** de la tool contra el contexto. El modelo solo pasa los argumentos. Es el patrón correcto: el LLM decide *qué* hacer; el código decide *si está permitido*.

email agent
- authenticates user
    - only then are they allowed into the "inbox"
    - dynamic tools and prompt on the condition of there being an email and password in state that match hardcoded
- checks "inbox"
    - email in tool
- sends emails
    - human in the loop

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: falta cargar local_model.py (raíz del repo) y definir el modelo local.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# import warnings
# warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*", category=UserWarning)
#
# from dotenv import load_dotenv
#
# load_dotenv()

In [2]:
# 🟢 ADAPTADO LOCAL
# Motivo: el notebook está en notebooks/module-3/ y local_model.py en la raíz del repo.
#         MODEL reemplaza a "gpt-5-nano" en todas las celdas (gemma4: soporta tools, el más rápido del lab).
import warnings
warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*", category=UserWarning)

import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model, OLLAMA_URL

MODEL = get_model("gemma4:latest")

### 📘 Contexto con las credenciales de prueba
Valores por defecto: `julie@example.com` / `password123`. Son datos de juguete del curso.

In [3]:
# Sin cambios
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

### 📘 State extendido
`authenticated: bool` empieza vacío (no existe → `request.state.get("authenticated")` devuelve `None`, que cuenta como falso).

In [4]:
# Sin cambios
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

### 📘 Las 3 tools
- `check_inbox` y `send_email` son falsas (texto fijo).
- `authenticate` compara con `runtime.context` y **escribe el state** con `Command(update=...)`. Debe incluir un `ToolMessage` con el mismo `tool_call_id`; si no, el modelo queda esperando el resultado de la tool.

In [5]:
# Sin cambios
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage(
                "Successfully authenticated", 
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed", 
                tool_call_id=runtime.tool_call_id)]
        })

### 📘 Dynamic tools según el state
Aquí el criterio es el **state** (`request.state.get("authenticated")`), no el contexto: cambia **durante** la conversación.

In [6]:
# Sin cambios
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Allow read inbox and send email tools only if user provides correct email and password"""

    authenticated = request.state.get("authenticated")
    
    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools) 
    return handler(request)

>Note: the prompts were modified since filming to constrain the model to more reliably match the filmed sequence. You may still experience different responses from the model, which is expected. You may need to modify the human message to provide appropriate responses.

### 📘 Dynamic prompt según el state
Dos prompts: uno para autenticar y otro para trabajar con la bandeja. El autenticado le dice al modelo que su **primer paso** es revisar el inbox.

In [7]:
# Sin cambios
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = """You are a helpful assistant that can check the inbox and send emails. 
Your first step after authentication is to check the inbox."""
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

### 📘 El agente completo
Fíjate en el orden de `middleware=[...]`: primero filtra tools, después arma el prompt y al final HITL. Los `wrap_model_call` se anidan en ese orden (el primero envuelve a los siguientes).

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent
# from langgraph.checkpoint.memory import InMemorySaver
# from langchain.agents.middleware import HumanInTheLoopMiddleware
#
# agent = create_agent(
#     "gpt-5-nano",
#     tools=[authenticate, check_inbox, send_email],
#     checkpointer=InMemorySaver(),
#     state_schema=AuthenticatedState,
#     context_schema=EmailContext,
#     middleware=[
#         dynamic_tool_call, 
#         dynamic_prompt,
#         HumanInTheLoopMiddleware(
#             interrupt_on={
#                 "authenticate": False,
#                 "check_inbox": False,
#                 "send_email": True,
#             })
#         ]
#     )
#

In [8]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" es API de pago.
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    MODEL,
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call, 
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
        ]
    )


### 📘 Turno 1: autenticación
Mandamos las credenciales como texto. Esperado: el modelo llama `authenticate` → *"Successfully authenticated"* → con las tools nuevas, revisa la bandeja y cuenta el correo de Jane.

In [9]:
# Sin cambios
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="julie@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

I see an email from Jane at jane@example.com asking if you'd like to grab coffee next week. Would you like me to reply to her? If so, what would you like the subject and body of the reply to be?


In [10]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

julie@example.com, password123
================================== Ai Message ==================================
Tool Calls:
  authenticate (1f55802f-d52f-4e49-a38b-44abfaf3e26a)
 Call ID: 1f55802f-d52f-4e49-a38b-44abfaf3e26a
  Args:
    email: julie@example.com
    password: password123
================================= Tool Message =================================
Name: authenticate

Successfully authenticated
================================== Ai Message ==================================
Tool Calls:
  check_inbox (789d364a-ffcd-4551-b3ea-2b8c173c6d2f)
 Call ID: 789d364a-ffcd-4551-b3ea-2b8c173c6d2f
  Args:
================================= Tool Message =================================
Name: check_inbox


    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    
================================== Ai Message ===============

### 📘 Turno 2: pedir la respuesta
Mismo `thread_id` → el agente recuerda que ya está autenticado (state en el checkpointer). Redacta y pide `send_email` → **pausa**. El cuerpo del correo está en `__interrupt__` (ver la tabla de "cajas dentro de cajas" en 3.3).

In [11]:
# Sin cambios

response = agent.invoke(
    {"messages": [HumanMessage(content="any draft is fine. don't check back.")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

In [12]:
# Sin cambios
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi Jane,

That sounds lovely! I'd love to grab coffee next week. Let me know what day and time works best for you.

Best,
Julie


In [13]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

julie@example.com, password123
================================== Ai Message ==================================
Tool Calls:
  authenticate (1f55802f-d52f-4e49-a38b-44abfaf3e26a)
 Call ID: 1f55802f-d52f-4e49-a38b-44abfaf3e26a
  Args:
    email: julie@example.com
    password: password123
================================= Tool Message =================================
Name: authenticate

Successfully authenticated
================================== Ai Message ==================================
Tool Calls:
  check_inbox (789d364a-ffcd-4551-b3ea-2b8c173c6d2f)
 Call ID: 789d364a-ffcd-4551-b3ea-2b8c173c6d2f
  Args:
================================= Tool Message =================================
Name: check_inbox


    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    
================================== Ai Message ===============

### 📘 Reanudar
`approve` envía el correo. Prueba también `reject` con un motivo, o `edit` con otro cuerpo, como en 3.3.

In [14]:
# Sin cambios
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

I have sent a reply to Jane. Is there anything else I can help you with?


In [15]:
# Sin cambios
from pprint import pprint

pprint(response)

{'authenticated': True,
 'messages': [HumanMessage(content='julie@example.com, password123', additional_kwargs={}, response_metadata={}, id='f0ca8b49-1522-4315-a6f2-07b1c4ab54ca'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:latest', 'created_at': '2026-10-05T22:11:31.4981301Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3067622500, 'load_duration': 6751700, 'prompt_eval_count': 91, 'prompt_eval_duration': 1798681000, 'eval_count': 27, 'eval_duration': 1251593000, 'logprobs': None, 'model_name': 'gemma4:latest', 'model_provider': 'ollama'}, id='lc_run--01a10e1f-33cb-7671-95f3-efd5bd4cdeac-0', tool_calls=[{'name': 'authenticate', 'args': {'email': 'julie@example.com', 'password': 'password123'}, 'id': '1f55802f-d52f-4e49-a38b-44abfaf3e26a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 91, 'output_tokens': 27, 'total_tokens': 118}),
              ToolMessage(content='Successfully authenticated'

In [16]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

julie@example.com, password123
================================== Ai Message ==================================
Tool Calls:
  authenticate (1f55802f-d52f-4e49-a38b-44abfaf3e26a)
 Call ID: 1f55802f-d52f-4e49-a38b-44abfaf3e26a
  Args:
    email: julie@example.com
    password: password123
================================= Tool Message =================================
Name: authenticate

Successfully authenticated
================================== Ai Message ==================================
Tool Calls:
  check_inbox (789d364a-ffcd-4551-b3ea-2b8c173c6d2f)
 Call ID: 789d364a-ffcd-4551-b3ea-2b8c173c6d2f
  Args:
================================= Tool Message =================================
Name: check_inbox


    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    
================================== Ai Message ===============